In [0]:
from pyspark.sql.functions import col, explode

LEVER_RAW_PATH = (
    "s3://job-market-analysis-pipeline/"
    "raw-data/lever/"
    "ingestion_date=2026-10-06/*.json"
)

lever_raw = spark.read.json(LEVER_RAW_PATH)

print("Lever RAW data loaded successfully")
print("Raw records:", lever_raw.count())

Lever RAW data loaded successfully
Raw records: 5


In [0]:
lever_raw.printSchema()

root
 |-- company: string (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- job_count: long (nullable = true)
 |-- jobs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- additional: string (nullable = true)
 |    |    |-- additionalPlain: string (nullable = true)
 |    |    |-- applyUrl: string (nullable = true)
 |    |    |-- categories: struct (nullable = true)
 |    |    |    |-- allLocations: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |-- commitment: string (nullable = true)
 |    |    |    |-- department: string (nullable = true)
 |    |    |    |-- level: string (nullable = true)
 |    |    |    |-- location: string (nullable = true)
 |    |    |    |-- team: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- createdAt: long (nullable = true)
 |    |    |-- description: string 

In [0]:
display(lever_raw.limit(5))

company,ingestion_date,ingestion_timestamp,job_count,jobs,site,source


In [0]:
lever_bronze = (
    lever_raw
    .select(
        col("source"),
        col("company"),
        col("site"),
        col("ingestion_timestamp"),
        col("ingestion_date"),
        explode(col("jobs")).alias("job")
    )
)

In [0]:
display(lever_bronze.limit(5))

source company site ingestion_timestamp ingestion_date job lever Palantir palantir 2026-10-06T12:52:06.606003+00:00 2026-10-06 List( Palantir is particularly interested in applicants who have a current right to work in Singapore, and we encourage Singapore citizens to apply. Life at Palantir 
   
 We want every Palantirian to achieve their best outcomes, that’s why we celebrate individuals’ strengths, skills, and interests, from your first interview to your longterm growth, rather than rely on traditional career ladders. Paying attention to the needs of our community enables us to optimize our opportunities to grow and helps ensure many pathways to success at Palantir. Promoting health and well-being across all areas of Palantirians’ lives is just one of the ways we’re investing in our community. Learn more at Life at Palantir and note that our offerings may vary by region. 
   
 In keeping consistent with Palantir’s values and culture, we believe employees are “better together” and in-person work affords the opportunity for more creative outcomes. Therefore, we encourage employees to work from our offices to foster connectivity and innovation. Many teams do offer hybrid options (WFH a day or two a week), allowing our employees to strike the right trade-off for their personal productivity. Based on business need, there are a few roles that allow for “Remote” work on an exceptional basis. If you are applying for one of these roles, you must work from the city and or country in which you are employed. If the posting is specified as Onsite, you are required to work from an office. 
   
 If you want to empower the world's most important institutions, you belong here. Palantir values excellence regardless of background. We are committed to making the application and hiring process accessible to everyone and will provide a reasonable accommodation for those living with a disability. If you need an accommodation for the application or hiring process , please reach out and let us know how we can help. 
   
 Please note that you will never be asked to submit a payment or share financial information to participate in our interview process. If you suspect that you've been contacted by a scammer, we recommend you cease all communication with the individual and consider reporting them to the relevant authorities, such as the US FBI Internet Crime Complaint Center (IC3) . , Palantir is particularly interested in applicants who have a current right to work in Singapore, and we encourage Singapore citizens to apply.


Life at Palantir
 
We want every Palantirian to achieve their best outcomes, that’s why we celebrate individuals’ strengths, skills, and interests, from your first interview to your longterm growth, rather than rely on traditional career ladders. Paying attention to the needs of our community enables us to optimize our opportunities to grow and helps ensure many pathways to success at Palantir. Promoting health and well-being across all areas of Palantirians’ lives is just one of the ways we’re investing in our community. Learn more at Life at Palantir and note that our offerings may vary by region.
 
In keeping consistent with Palantir’s values and culture, we believe employees are “better together” and in-person work affords the opportunity for more creative outcomes. Therefore, we encourage employees to work from our offices to foster connectivity and innovation. Many teams do offer hybrid options (WFH a day or two a week), allowing our employees to strike the right trade-off for their personal productivity. Based on business need, there are a few roles that allow for “Remote” work on an exceptional basis. If you are applying for one of these roles, you must work from the city and or country in which you are employed. If the posting is specified as Onsite, you are required to work from an office.
 
If you want to empower the world's most important institutions, you belong here. Palantir values excellence regardless of backgro

In [0]:
lever_bronze = (
    lever_bronze
    .select(
        # Ingestion / lineage metadata
        col("source"),
        col("company"),
        col("site"),
        col("ingestion_timestamp"),
        col("ingestion_date"),

        # Job identifiers
        col("job.id").alias("job_id"),
        col("job.text").alias("job_title"),

        # Job categories
        col("job.categories.location").alias("location"),
        col("job.categories.allLocations").alias("all_locations"),
        col("job.categories.department").alias("department"),
        col("job.categories.team").alias("team"),
        col("job.categories.level").alias("job_level"),
        col("job.categories.commitment").alias("employment_type"),

        # Job description
        col("job.description").alias("description"),
        col("job.descriptionPlain").alias("description_plain"),

        # Job details
        col("job.country").alias("country"),
        col("job.workplaceType").alias("workplace_type"),

        # Salary
        col("job.salaryRange").alias("salary_range"),
        col("job.salaryDescription").alias("salary_description"),

        # Dates
        col("job.createdAt").alias("created_at"),

        # URLs
        col("job.hostedUrl").alias("job_url"),
        col("job.applyUrl").alias("apply_url")
    )
)

In [0]:
lever_bronze.printSchema()

root
 |-- source: string (nullable = true)
 |-- company: string (nullable = true)
 |-- site: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- job_id: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- location: string (nullable = true)
 |-- all_locations: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- department: string (nullable = true)
 |-- team: string (nullable = true)
 |-- job_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- description_plain: string (nullable = true)
 |-- country: string (nullable = true)
 |-- workplace_type: string (nullable = true)
 |-- salary_range: struct (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- interval: string (nullable = true)
 |    |-- max: long (nullable = true)
 |    |-- min: long (nullable = true)
 |-- salary_descripti

In [0]:
display(lever_bronze.limit(10))

source company site ingestion_timestamp ingestion_date job_id job_title location all_locations department team job_level employment_type description description_plain country workplace_type salary_range salary_description created_at job_url apply_url lever Palantir palantir 2026-10-06T12:52:06.606003+00:00 2026-10-06 6ed76ce8-4156-4b60-b120-403538bd66cd Administrative Business Partner Singapore, Singapore List(Singapore, Singapore) null Administrative null Full-time A World-Changing Company 
   
 Palantir builds the world’s leading software for data-driven decisions and operations. By bringing the right data to the people who need it, our platforms empower our partners to develop lifesaving drugs, forecast supply chain disruptions, locate missing children, and more. The Role 
   
 Our team of Administrative Business Partners does more than just support our leaders: we're the backbone of the busiest people at Palantir. We build positive relationships with the people we support and anticipate their needs without being asked. Our passion for helping others makes us an invaluable resource at Palantir! As an Administrative Business Partner, you will be handling a variety of professional responsibilities, including calendaring, travel, and expenses. You are very organized and thrive off of enabling the people we support to be as productive and impactful as possible. You'll demonstrate your excellent communication skills, and exercise tact and diplomacy in helping to manage relationships with internal and external senior team members at Palantir. In this role you'll also demonstrate good judgment and critical thinking by understanding competing priorities and actioning accordingly. A World-Changing Company
 
Palantir builds the world’s leading software for data-driven decisions and operations. By bringing the right data to the people who need it, our platforms empower our partners to develop lifesaving drugs, forecast supply chain disruptions, locate missing children, and more.


The Role
 
Our team of Administrative Business Partners does more than just support our leaders: we're the backbone of the busiest people at Palantir. We build positive relationships with the people we support and anticipate their needs without being asked. Our passion for helping others makes us an invaluable resource at Palantir!

As an Administrative Business Partner, you will be handling a variety of professional responsibilities, including calendaring, travel, and expenses. You are very organized and thrive off of enabling the people we support to be as productive and impactful as possible. You'll demonstrate your excellent communication skills, and exercise tact and diplomacy in helping to manage relationships with internal and external senior team members at Palantir. In this role you'll also demonstrate good judgment and critical thinking by understanding competing priorities and actioning accordingly.
 SG hybrid null null 1786469891368 https://jobs.lever.co/palantir/6ed76ce8-4156-4b60-b120-403538bd66cd https://jobs.lever.co/palantir/6ed76ce8-4156-4b60-b120-403538bd66cd/apply lever Palantir palantir 2026-10-06T12:52:06.606003+00:00 2026-10-06 ac978161-6f46-4f6b-ad9e-a258e642751c Administrative Business Partner London, United Kingdom List(London, United Kingdom) null Administrative null Full-time A World-Changing Company 
   
 Palantir builds the world’s leading software for data-driven decisions and operations. By bringing the right data to the people who need it, our platforms empower our partners to develop lifesaving drugs, forecast supply chain disruptions, locate missing children, and more. The Role 
   
 Our team of Administrative Business Partners does more than just support our leaders: we're the backbone of the busiest people at Palantir. We build positive relationships with the people we support and anticipate their needs without being asked. Our passion for helping others makes us an invaluable resource at Palantir! As an Administrative Bu

In [0]:
print("Lever Bronze records:", lever_bronze.count())

Lever Bronze records: 638


In [0]:
lever_bronze.groupBy("company").count().orderBy("company").show()

+--------+-----+
| company|count|
+--------+-----+
|   DEUNA|   17|
|Palantir|  316|
| Spotify|   76|
|    Zoox|  229|
+--------+-----+



In [0]:
from pyspark.sql.functions import count, countDistinct

lever_bronze.select(
    count("*").alias("total_records"),
    countDistinct("job_id").alias("unique_job_ids")
).show()

+-------------+--------------+
|total_records|unique_job_ids|
+-------------+--------------+
|          638|           638|
+-------------+--------------+



In [0]:
from pyspark.sql.functions import sum

lever_bronze.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in lever_bronze.columns
]).show()

+------+-------+----+-------------------+--------------+------+---------+--------+-------------+----------+----+---------+---------------+-----------+-----------------+-------+--------------+------------+------------------+----------+-------+---------+
|source|company|site|ingestion_timestamp|ingestion_date|job_id|job_title|location|all_locations|department|team|job_level|employment_type|description|description_plain|country|workplace_type|salary_range|salary_description|created_at|job_url|apply_url|
+------+-------+----+-------------------+--------------+------+---------+--------+-------------+----------+----+---------+---------------+-----------+-----------------+-------+--------------+------------+------------------+----------+-------+---------+
|     0|      0|   0|                  0|             0|     0|        0|       0|            0|       333|   0|      612|             14|          0|                0|      0|             0|         422|               444|         0|      0

In [0]:
(
    lever_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("job_market.bronze.lever_jobs")
)

In [0]:
spark.sql("""
    SELECT *
    FROM job_market.bronze.lever_jobs
    LIMIT 10
""").display()

source company site ingestion_timestamp ingestion_date job_id job_title location all_locations department team job_level employment_type description description_plain country workplace_type salary_range salary_description created_at job_url apply_url lever DEUNA deuna 2026-10-06T12:52:06.606003+00:00 2026-10-06 26b86395-02b2-4540-8d97-6c648f3fab79 AI Platform Tech Lead San Francisco List(San Francisco) null Technology null Full time About DEUNA 
 DEUNA is a payments infrastructure platform that helps enterprise merchants across Latin America, the US, and Europe optimize and orchestrate their entire payment stack. We combine payment routing intelligence, AI-driven optimization, and a composable checkout experience to help companies increase revenue and reduce operational complexity at scale. We are backed by leading investors and processing billions of dollars in annual transaction volume. 
 About the Role 
 DEUNA is a payments infrastructure company powering enterprise commerce across Latin America, the US, and Europe. We operate at the intersection of high-volume payment orchestration and applied AI — building intelligent systems that optimize authorization rates, reduce costs, and automate complex payment workflows for some of the largest merchants in the world. 
   
 We are looking for a Staff/Principal-level AI Platform Tech Lead to own the full technical stack behind our AI payment intelligence and digital workforce products — from ML model training through production routing integration. This is a hands-on leadership role: you will set the architecture, write the code, and grow the team. About DEUNA
DEUNA is a payments infrastructure platform that helps enterprise merchants across Latin America, the US, and Europe optimize and orchestrate their entire payment stack. We combine payment routing intelligence, AI-driven optimization, and a composable checkout experience to help companies increase revenue and reduce operational complexity at scale. We are backed by leading investors and processing billions of dollars in annual transaction volume.
About the Role
DEUNA is a payments infrastructure company powering enterprise commerce across Latin America, the US, and Europe. We operate at the intersection of high-volume payment orchestration and applied AI — building intelligent systems that optimize authorization rates, reduce costs, and automate complex payment workflows for some of the largest merchants in the world.
 
We are looking for a Staff/Principal-level AI Platform Tech Lead to own the full technical stack behind our AI payment intelligence and digital workforce products — from ML model training through production routing integration. This is a hands-on leadership role: you will set the architecture, write the code, and grow the team.
 US hybrid null null 1779478818104 https://jobs.lever.co/deuna/26b86395-02b2-4540-8d97-6c648f3fab79 https://jobs.lever.co/deuna/26b86395-02b2-4540-8d97-6c648f3fab79/apply lever DEUNA deuna 2026-10-06T12:52:06.606003+00:00 2026-10-06 a1d9d480-6b02-448e-99a1-0e72e36b31d0 Backend Engineer Bogotá, Colombia List(Bogotá, Colombia, Ciudad de México, Mexico, Argentina) null Technology null Full time About DEUNA  🧡 
   
 DEUNA is a rapidly growing startup revolutionizing global commerce with ATHIA, our AI-powered orchestration and payments platform that helps large enterprises boost approval rates, reduce costs, and unlock new revenue. Built by the team behind DEUNA—the fastest-growing Commerce OS in Latin America—ATHIA combines payment intelligence, checkout optimization, and data orchestration in one powerful solution. 
   
 With deep integrations across 300+ PSPs and alternative payment methods, and over 20% of Mexico’s digital economy running through our platform, we simplify global payments through a single integration and centralized reconciliation. 
 We are a rapidly growing startup expanding into the U.S. to meet the urgent needs of large retailers, marketplaces, airlines, and QSRs. Join

In [0]:
spark.sql("""
    SELECT
        company,
        COUNT(*) AS job_count
    FROM job_market.bronze.lever_jobs
    GROUP BY company
    ORDER BY job_count DESC
""").display()

company,job_count
Palantir,316
Zoox,229
Spotify,76
DEUNA,17


In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_jobs
    FROM job_market.bronze.lever_jobs
""").show()

+----------+
|total_jobs|
+----------+
|       638|
+----------+

